# MedPredict s.r.l. — Previsione della Progressione del Diabete

**Caso d'uso aziendale:** Previsione della progressione del diabete in pazienti a rischio.

**Obiettivo:** Sviluppare un modello di regressione predittiva che, utilizzando dati clinici dei pazienti, fornisca previsioni accurate sulla progressione della malattia nel tempo.

**Dataset:** Diabetes dataset (scikit-learn) — 442 pazienti, 10 variabili cliniche, target quantitativo di progressione della malattia.

---

### Variabili indipendenti
| Feature | Descrizione |
|---------|-------------|
| age | Età del paziente |
| sex | Genere |
| bmi | Indice di massa corporea |
| bp | Pressione sanguigna media |
| s1 | Colesterolo sierico totale |
| s2 | Lipoproteine a bassa densità (LDL) |
| s3 | Lipoproteine ad alta densità (HDL) |
| s4 | Rapporto colesterolo totale / HDL |
| s5 | Trigliceridi (log) |
| s6 | Livello di glicemia |

**Target:** misura quantitativa della progressione del diabete a un anno dalla raccolta dati.

# Moduli

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (
    LinearRegression,
    Ridge,    RidgeCV,
    Lasso,    LassoCV,
    ElasticNet, ElasticNetCV
)
from sklearn.metrics import mean_squared_error, r2_score

# Metodi comuni

Visualizza la matrice di correlazione come heatmap, con i coefficienti di Pearson annotati in ciascuna cella.

In [ ]:
def print_correlation_matrix(df):
    """
    Print correlation matrix as a heatmap.

    Args:
        df (DataFrame): Input DataFrame.

    Returns:
        None
    """
    plt.figure(figsize=(12, 10), dpi=100)
    sns.set_style("whitegrid")
    sns.heatmap(
        df.corr(),
        cmap="crest",
        cbar=True,
        square=True,
        yticklabels=df.columns,
        xticklabels=df.columns,
        annot=True,
        annot_kws={'size': 10},
        fmt='.2f'
    )
    plt.tight_layout()
    plt.show()

Standardizza le feature con `StandardScaler`: fit sul training set, transform su entrambi i set. Evita data leakage e restituisce lo scaler fittato per il riutilizzo in produzione.

In [ ]:
def scale_features(X_train, X_test):
    """
    Scale features using StandardScaler.

    Args:
        X_train (array-like): Training data.
        X_test (array-like): Test data.

    Returns:
        Tuple: (X_train_scaled, X_test_scaled, scaler)
    """
    ss = StandardScaler()
    X_train_scaled = ss.fit_transform(X_train)
    X_test_scaled = ss.transform(X_test)
    return X_train_scaled, X_test_scaled, ss

Scatter plot predetto vs reale con retta di riferimento (previsione perfetta). L'allineamento dei punti sulla diagonale misura la qualità del modello.

In [ ]:
def plot_predictions(y_test, y_pred, title=""):
    """
    Plot predicted vs actual values.

    Args:
        y_test (array-like): True values.
        y_pred (array-like): Predicted values.
        title (str): Plot title.

    Returns:
        None
    """
    plt.figure(figsize=(6, 5), dpi=100)
    plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidths=0.4)
    lim = [min(y_test.min(), y_pred.min()) - 5, max(y_test.max(), y_pred.max()) + 5]
    plt.plot(lim, lim, 'r--', lw=2, label='Previsione perfetta')
    plt.xlabel('Valori Reali')
    plt.ylabel('Valori Predetti')
    plt.title(f'Predetto vs Reale — {title}')
    plt.legend()
    plt.tight_layout()
    plt.show()

Calcola MSE, RMSE e R² su training e test, e mostra il grafico predetto vs reale.

In [ ]:
def evaluate_model(model_name, y_train, y_test, y_pred_train, y_pred_test):
    """
    Evaluate regression model performance.

    Args:
        model_name (str): Name of the model.
        y_train, y_test (array-like): True values.
        y_pred_train, y_pred_test (array-like): Predicted values.

    Returns:
        None
    """
    for split, y_true, y_pred in [("TRAIN", y_train, y_pred_train), ("TEST", y_test, y_pred_test)]:
        mse  = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_true, y_pred)
        print(f"{model_name} | {split}")
        print(f"  MSE  : {mse:.2f}")
        print(f"  RMSE : {rmse:.2f}")
        print(f"  R²   : {r2:.4f}")
        print()

    plot_predictions(y_test, y_pred_test, model_name)

Addestra il modello, valuta le prestazioni su training e test, e restituisce il modello addestrato con le metriche sul test set per la tabella comparativa.

In [ ]:
def build_model(X_train, X_test, y_train, y_test, model, model_name):
    """
    Build and evaluate a regression model.

    Args:
        X_train, X_test (array-like): Features.
        y_train, y_test (array-like): Target values.
        model: scikit-learn regression estimator.
        model_name (str): Label for output.

    Returns:
        Tuple: (trained model, metrics dict with R2 and RMSE on test set)
    """
    X_train_scaled, X_test_scaled, _ = scale_features(X_train, X_test)

    model.fit(X_train_scaled, y_train)

    y_pred_train = model.predict(X_train_scaled)
    y_pred_test  = model.predict(X_test_scaled)

    evaluate_model(model_name, y_train, y_test, y_pred_train, y_pred_test)

    metrics = {
        'R2':   round(r2_score(y_test, y_pred_test), 4),
        'RMSE': round(float(np.sqrt(mean_squared_error(y_test, y_pred_test))), 2)
    }
    return model, metrics

Visualizza l'andamento di MSE su training e validation al variare della dimensione del training set. Strumento principale per diagnosticare il tradeoff bias-varianza:
- curve entrambe alte con gap ridotto → **underfitting** (alto bias)
- gap elevato tra training e validation → **overfitting** (alta varianza)
- curve convergenti a valore basso → modello ben calibrato

In [ ]:
def plot_learning_curve(model, X, y, title="", cv=5):
    """
    Plot learning curve to diagnose bias-variance tradeoff.

    Args:
        model: scikit-learn estimator (already includes scaling if needed).
        X (array-like): Features.
        y (array-like): Target values.
        title (str): Plot title.
        cv (int): Number of cross-validation folds.

    Returns:
        None
    """
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        cv=cv,
        scoring='neg_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=-1
    )

    train_mse = -train_scores.mean(axis=1)
    val_mse   = -val_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_std   = val_scores.std(axis=1)

    plt.figure(figsize=(9, 5), dpi=100)
    plt.plot(train_sizes, train_mse, 'o-', color='steelblue', label='Training MSE')
    plt.fill_between(train_sizes,
                     train_mse - train_std,
                     train_mse + train_std,
                     alpha=0.15, color='steelblue')
    plt.plot(train_sizes, val_mse, 'o-', color='tomato', label='Validation MSE')
    plt.fill_between(train_sizes,
                     val_mse - val_std,
                     val_mse + val_std,
                     alpha=0.15, color='tomato')
    plt.xlabel('Dimensione Training Set')
    plt.ylabel('MSE')
    plt.title(f'Learning Curve — {title}')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Caricamento del Dataset

Carico il dataset Diabetes di scikit-learn e lo converto in un DataFrame Pandas aggiungendo la colonna `target`.

In [ ]:
diabetes = load_diabetes()

df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target

print(f"Dimensioni dataset: {df.shape}")
df.head()

## Statistiche descrittive

Statistiche descrittive delle variabili cliniche: media, deviazione standard, percentili e range.

In [ ]:
df.describe().round(3)

## Controllo valori nulli

Verifico l'assenza di valori nulli nel dataset.

In [ ]:
df.isnull().sum()

## Pre-processing

La variabile `sex` è già codificata numericamente da scikit-learn (+1/−1): nessuna operazione di encoding necessaria. Verifico anche la presenza di outlier con la regola IQR: poiché il dataset è clinico e di piccole dimensioni (442 record), gli outlier vengono mantenuti per non ridurre ulteriormente i dati disponibili.

In [ ]:
# Codifica sex: scikit-learn usa valori numerici (+1 / -1)
print("Valori unici di sex:", np.unique(df['sex'].values))

# Outlier IQR su tutte le variabili
print("\nConteggio outlier per variabile (regola IQR, soglia 1.5):")
for col in df.columns:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f"  {col:>8}: {n_out}")

print("\nOutlier mantenuti: dataset clinico di piccole dimensioni (442 record).")
print("La rimozione ridurrebbe i dati disponibili e potrebbe eliminare casi clinicamente rilevanti.")

# Analisi Esplorativa dei Dati (EDA)

## Distribuzione della variabile target

Istogramma e boxplot del target. Una distribuzione approssimativamente normale è favorevole per la regressione lineare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

axes[0].hist(df['target'], bins=30, edgecolor='black', color='steelblue')
axes[0].set_xlabel('Progressione Diabete (target)')
axes[0].set_ylabel('Frequenza')
axes[0].set_title('Distribuzione del Target')

axes[1].boxplot(df['target'], vert=True)
axes[1].set_ylabel('Progressione Diabete (target)')
axes[1].set_title('Boxplot del Target')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

print(f"Media: {df['target'].mean():.2f} | Mediana: {df['target'].median():.2f} | Std: {df['target'].std():.2f}")

## Scatter plot: Feature vs Target

Relazione tra ciascuna feature e il target. Le variabili con scatter più allungato presentano correlazione lineare più forte con la progressione del diabete.

In [ ]:
feature_cols = diabetes.feature_names

fig, axes = plt.subplots(2, 5, figsize=(20, 8), dpi=100)
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].scatter(df[col], df['target'], alpha=0.5, edgecolors='k', linewidths=0.2, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Target')
    axes[i].set_title(f'{col} vs Target')

plt.suptitle('Relazione Feature-Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Matrice di correlazione — Dataset completo

Analisi della matrice di correlazione di Pearson. L'ultima riga/colonna mostra la correlazione di ciascuna feature con il target; valori elevati tra feature indicano possibile multicollinearità.

In [ ]:
print_correlation_matrix(df)

Correlazioni con il target ordinate per valore assoluto decrescente, per identificare rapidamente le feature più predittive.

In [ ]:
corr_with_target = df.corr()['target'].drop('target').sort_values(key=abs, ascending=False)

plt.figure(figsize=(9, 5), dpi=100)
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_with_target.values]
plt.bar(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Feature Clinica')
plt.ylabel('Correlazione con Target')
plt.title('Correlazione delle Feature con la Progressione del Diabete')
plt.tight_layout()
plt.show()

print(corr_with_target)

# Feature Engineering

## Analisi della multicollinearità

Dalla matrice emergono coppie di feature fortemente correlate: **s1/s2** (colesterolo totale e LDL) e **s3/s4** (HDL e rapporto colesterolo/HDL). La multicollinearità può destabilizzare i coefficienti della regressione semplice. Costruisco due versioni del dataset:
- **Dataset completo** (`df`): tutte e 10 le feature
- **Dataset ridotto** (`df2`): escluse le feature più ridondanti

Le feature con correlazione più debole verso il target e maggiore ridondanza reciproca sono **s1**, **s2** e **sex**. Creo `df2` rimuovendole come ipotesi da testare: i modelli regolarizzati gestiscono internamente la multicollinearità (Lasso può già azzerare s2 autonomamente). La scelta del dataset finale sarà guidata dalla tabella comparativa.

In [ ]:
df2 = df.drop(['s1', 's2', 'sex'], axis=1)
print(f"Feature dataset completo:  {list(df.drop('target', axis=1).columns)}")
print(f"Feature dataset ridotto:   {list(df2.drop('target', axis=1).columns)}")

## Matrice di correlazione — Dataset ridotto

Verifico che il dataset ridotto presenti minore multicollinearità tra le feature rimaste.

In [ ]:
print_correlation_matrix(df2)

## Dataset per l'identificazione del modello

Creo le variabili `X` e `y` per entrambe le versioni del dataset, da usare nel confronto dei modelli.

In [ ]:
X   = df.drop('target', axis=1).values
y   = df['target'].values

X_2 = df2.drop('target', axis=1).values
y_2 = df2['target'].values

print(f"Shape dataset completo:  X={X.shape}, y={y.shape}")
print(f"Shape dataset ridotto:   X={X_2.shape}, y={y_2.shape}")

# Regression Models

Testo quattro varianti della regressione lineare con strategie di regolarizzazione crescente. L'alpha ottimale è selezionato automaticamente tramite cross-validation su griglia log-uniforme.

| Modello | Regolarizzazione | Ottimizzazione alpha |
|---------|-----------------|---------------------|
| Linear Regression | Nessuna (baseline) | — |
| Ridge | L2 | `RidgeCV` (k=5) |
| Lasso | L1 | `LassoCV` (k=5) |
| ElasticNet | L1 + L2 | `ElasticNetCV` (k=5, griglia alpha × l1_ratio) |

Ogni modello è testato sul **dataset completo** e sul **dataset ridotto** (split 80/20, `random_state=0`).

Dizionario per raccogliere le metriche di tutti i modelli per la tabella comparativa finale.

In [ ]:
results = {}

## Linear Regression (baseline)

Regressione lineare senza regolarizzazione: baseline di riferimento per i modelli successivi.

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

_, results['Linear Regression — completo'] = build_model(
    X_train, X_test, y_train, y_test,
    LinearRegression(),
    "Linear Regression — completo"
)

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

_, results['Linear Regression — ridotto'] = build_model(
    X_train, X_test, y_train, y_test,
    LinearRegression(),
    "Linear Regression — ridotto"
)

### Conclusioni Linear Regression
La regressione lineare mostra già un R² positivo, confermando il potere predittivo delle feature cliniche. Costituisce il termine di confronto per valutare il beneficio della regolarizzazione.

## Ridge Regression (L2)

Ridge penalizza la norma L2 dei coefficienti, riducendo la sensibilità alla multicollinearità. Il parametro `alpha` è ottimizzato automaticamente con `RidgeCV` (k=5, griglia log-uniforme su [10⁻³, 10³]).

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

ridge_full, results['Ridge — completo'] = build_model(
    X_train, X_test, y_train, y_test,
    RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5),
    "Ridge CV — completo"
)
print(f"  → Alpha ottimale selezionato da RidgeCV: {ridge_full.alpha_:.4f}")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

ridge_ridotto, results['Ridge — ridotto'] = build_model(
    X_train, X_test, y_train, y_test,
    RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5),
    "Ridge CV — ridotto"
)
print(f"  → Alpha ottimale selezionato da RidgeCV: {ridge_ridotto.alpha_:.4f}")

### Conclusioni Ridge
La regolarizzazione L2 migliora le prestazioni rispetto alla baseline. Ridge performa **meglio sul dataset completo**: distribuisce il peso tra feature correlate (s1, s2) invece di doverle rimuovere. Rimuoverle manualmente comporta perdita di informazione, a differenza di quanto accade per la regressione lineare semplice.

## Lasso Regression (L1)

Lasso penalizza la norma L1: può portare alcuni coefficienti esattamente a zero, effettuando selezione automatica delle feature. Il parametro `alpha` è ottimizzato con `LassoCV` (k=5).

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

lasso_full, results['Lasso — completo'] = build_model(
    X_train, X_test, y_train, y_test,
    LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, max_iter=10000),
    "Lasso CV — completo"
)
print(f"  → Alpha ottimale selezionato da LassoCV: {lasso_full.alpha_:.4f}")
print()
print("Coefficienti Lasso (dataset completo):")
for name, coef in zip(diabetes.feature_names, lasso_full.coef_):
    tag = '  ← azzerato' if coef == 0 else ''
    print(f"  {name:>5}: {coef:8.4f}{tag}")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

lasso_ridotto, results['Lasso — ridotto'] = build_model(
    X_train, X_test, y_train, y_test,
    LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, max_iter=10000),
    "Lasso CV — ridotto"
)
print(f"  → Alpha ottimale selezionato da LassoCV: {lasso_ridotto.alpha_:.4f}")
print()
print("Coefficienti Lasso (dataset ridotto):")
for name, coef in zip(df2.drop('target', axis=1).columns, lasso_ridotto.coef_):
    tag = '  ← azzerato' if coef == 0 else ''
    print(f"  {name:>5}: {coef:8.4f}{tag}")

### Conclusioni Lasso
Lasso effettua selezione automatica delle feature: sul dataset completo azzera già s2 (elevata multicollinearità), rendendo parzialmente ridondante la rimozione manuale. Sul dataset ridotto azzera anche age, s4 e s6 — ridondanti rispetto a s5 (r=0.62 tra le due).

## ElasticNet (L1 + L2)

ElasticNet combina penalità L1 e L2. Entrambi `alpha` e `l1_ratio` sono ottimizzati automaticamente con `ElasticNetCV` (k=5, griglia bidimensionale alpha × l1_ratio).

### Dataset completo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

en_full, results['ElasticNet — completo'] = build_model(
    X_train, X_test, y_train, y_test,
    ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                 alphas=np.logspace(-4, 1, 50),
                 cv=5, max_iter=10000),
    "ElasticNet CV — completo"
)
print(f"  → Alpha ottimale: {en_full.alpha_:.4f}  |  l1_ratio ottimale: {en_full.l1_ratio_:.2f}")

### Dataset ridotto

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2, test_size=0.2, random_state=0)

en_ridotto, results['ElasticNet — ridotto'] = build_model(
    X_train, X_test, y_train, y_test,
    ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                 alphas=np.logspace(-4, 1, 50),
                 cv=5, max_iter=10000),
    "ElasticNet CV — ridotto"
)
print(f"  → Alpha ottimale: {en_ridotto.alpha_:.4f}  |  l1_ratio ottimale: {en_ridotto.l1_ratio_:.2f}")

### Conclusioni ElasticNet
L'`l1_ratio` ottimale è **1.0 su entrambi i dataset**: la componente L2 non contribuisce e ElasticNet degenera in puro Lasso, con risultati identici. Questo conferma che la regolarizzazione ottimale per questo problema è **L1 pura**, e Lasso è il modello naturale da scegliere come finale.

# Conclusioni Finali e Scelta del Modello

## Tabella comparativa — tutti i modelli

Raccolgo le metriche delle 8 configurazioni testate e le ordino per R² decrescente. Il modello migliore è quello con **R² più alto** e **RMSE più basso**.

In [ ]:
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Modello', 'R2', 'RMSE']
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)

print("Confronto metriche sul TEST SET (ordinate per R² decrescente):")
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

colors_bar = ['steelblue' if '— ridotto' in m else 'lightsteelblue' for m in results_df['Modello']]

axes[0].barh(results_df['Modello'], results_df['R2'], color=colors_bar, edgecolor='black')
axes[0].set_xlabel('R² (Test Set)')
axes[0].set_title('R² — Confronto Modelli')
axes[0].axvline(x=results_df['R2'].max(), color='red', linestyle='--', linewidth=0.8)

axes[1].barh(results_df['Modello'], results_df['RMSE'], color=colors_bar, edgecolor='black')
axes[1].set_xlabel('RMSE (Test Set)')
axes[1].set_title('RMSE — Confronto Modelli')
axes[1].axvline(x=results_df['RMSE'].min(), color='red', linestyle='--', linewidth=0.8)

plt.suptitle('Barre scure = dataset ridotto  |  Barre chiare = dataset completo', fontsize=10)
plt.tight_layout()
plt.show()

best = results_df.iloc[0]
print(f"\nModello con R² migliore: {best['Modello']}")
print(f"  R² = {best['R2']}  |  RMSE = {best['RMSE']}")

## Dataset finale

Dataset finale scelto in modo data-driven dalla tabella comparativa. I modelli regolarizzati gestiscono internamente la multicollinearità — la scelta ottimale emerge dai dati, non da un'assunzione a priori.

In [ ]:
# Scelta data-driven: il dataset migliore emerge dalla tabella comparativa
best_model_name = results_df.iloc[0]['Modello']
use_ridotto = '— ridotto' in best_model_name
dataset_final = df2.copy() if use_ridotto else df.copy()
dataset_label = 'ridotto (df2)' if use_ridotto else 'completo (df)'

print(f"Modello con R² migliore: {best_model_name}")
print(f"Dataset selezionato:     {dataset_label}")
print(f"Feature del modello finale: {list(dataset_final.drop('target', axis=1).columns)}")
print(f"Shape: {dataset_final.shape}")
dataset_final.head()

## Modello finale

**Lasso** come modello finale: ElasticNet ha convergito a `l1_ratio=1.0` su entrambi i dataset, confermando che la regolarizzazione L1 è quella ottimale. Riaddestriamo `LassoCV` sul training set finale e conserviamo lo scaler fittato per l'applicazione in produzione.

In [ ]:
X_final = dataset_final.drop('target', axis=1).values
y_final = dataset_final['target'].values

print(f"Shape X: {X_final.shape}")
print(f"Shape y: {y_final.shape}")

In [ ]:
X_final = dataset_final.drop('target', axis=1).values
y_final = dataset_final['target'].values
feature_names_final = list(dataset_final.drop('target', axis=1).columns)

print(f"Shape X: {X_final.shape}")
print(f"Shape y: {y_final.shape}")

X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=0)

# Scaling esplicito: conserviamo lo scaler per l'export
X_train_scaled, X_test_scaled, scaler_export = scale_features(X_train, X_test)

# Alpha ottimizzato tramite cross-validation sul training set finale
lasso_cv_final = LassoCV(alphas=np.logspace(-4, 1, 50), cv=5, max_iter=10000)
lasso_cv_final.fit(X_train_scaled, y_train)
optimal_alpha = lasso_cv_final.alpha_
print(f"\nAlpha ottimale per il modello finale: {optimal_alpha:.4f}")

final_model = Lasso(alpha=optimal_alpha, max_iter=10000)
final_model.fit(X_train_scaled, y_train)

y_pred_train = final_model.predict(X_train_scaled)
y_pred_test  = final_model.predict(X_test_scaled)

evaluate_model("Lasso — Modello Finale", y_train, y_test, y_pred_train, y_pred_test)

print("\nCoefficienti Lasso — Modello Finale:")
for name, coef in zip(feature_names_final, final_model.coef_):
    tag = '  ← azzerato' if coef == 0 else ''
    print(f"  {name:>5}: {coef:8.4f}{tag}")

## Cross-Validation

K-fold cross-validation (k=5) sull'intero dataset selezionato. Rispetto al singolo hold-out, fornisce una stima più robusta della generalizzazione. Il gap tra R² medio in CV e R² hold-out indica quanto la singola divisione possa essere ottimistica o pessimistica.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=optimal_alpha, max_iter=10000))
])

cv_scores = cross_val_score(pipeline, X_final, y_final, cv=5, scoring='r2')

print(f"R² per fold: {np.round(cv_scores, 4)}")
print(f"R² medio:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"R² hold-out: {r2_score(y_test, y_pred_test):.4f}")
print(f"Δ (CV − hold-out): {cv_scores.mean() - r2_score(y_test, y_pred_test):+.4f}")

plt.figure(figsize=(8, 4), dpi=100)
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Media R²={cv_scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('R²')
plt.title('K-Fold Cross-Validation R² — Lasso Modello Finale')
plt.legend()
plt.tight_layout()
plt.show()

## Learning Curve — Analisi del Tradeoff Bias-Varianza

Andamento di MSE su training e validation al crescere della dimensione del training set:
- curve entrambe alte con gap ridotto → **underfitting**
- gap elevato → **overfitting**
- curve convergenti a valore basso → modello ben calibrato

In [ ]:
plot_learning_curve(pipeline, X_final, y_final, title=f"Lasso — {dataset_label}", cv=5)

Confronto tra la learning curve di Lasso e la baseline (Linear Regression) per visualizzare l'effetto della regolarizzazione L1 sul tradeoff bias-varianza.

In [ ]:
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=100)

for ax, pipe, title in [
    (axes[0], pipeline_lr, "Linear Regression (baseline)"),
    (axes[1], pipeline,    f"Lasso (L1 — alpha={optimal_alpha:.4f})")
]:
    train_sizes, train_scores, val_scores = learning_curve(
        pipe, X_final, y_final,
        cv=5, scoring='neg_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    train_mse = -train_scores.mean(axis=1)
    val_mse   = -val_scores.mean(axis=1)

    ax.plot(train_sizes, train_mse, 'o-', color='steelblue', label='Training MSE')
    ax.plot(train_sizes, val_mse,   'o-', color='tomato',    label='Validation MSE')
    ax.set_xlabel('Dimensione Training Set')
    ax.set_ylabel('MSE')
    ax.set_title(f'Learning Curve — {title}')
    ax.legend()

plt.tight_layout()
plt.show()

## Esportazione del Modello

Esporto il modello finale con `pickle` insieme allo scaler e ai metadati per la produzione. Lo stesso scaler fittato su `X_train` garantisce pre-processing coerente in fase di inferenza, senza rischi di disallineamento.

In [ ]:
model_artifact = {
    'model':        final_model,
    'scaler':       scaler_export,          # stesso scaler usato per il training
    'features':     feature_names_final,
    'model_name':   f'Lasso (alpha={optimal_alpha:.4f})',
    'optimal_alpha': optimal_alpha,
    'test_r2':      float(r2_score(y_test, y_pred_test)),
    'test_rmse':    float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
}

with open('medpredict_diabetes_model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print("Modello salvato in: medpredict_diabetes_model.pkl")
print(f"Modello:       {model_artifact['model_name']}")
print(f"Feature:       {model_artifact['features']}")
print(f"R² Test:       {model_artifact['test_r2']:.4f}")
print(f"RMSE Test:     {model_artifact['test_rmse']:.2f}")

In [ ]:
with open('medpredict_diabetes_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

y_pred_check = loaded['model'].predict(loaded['scaler'].transform(X_test))
print(f"Previsioni identiche dopo il ricaricamento: {np.allclose(y_pred_test, y_pred_check)}")
print("Modello pronto per la piattaforma sanitaria MedPredict.")

## Conclusioni Finali

- **Esplorazione**: le feature più predittive sono **bmi**, **s5** e **bp**; coppie con multicollinearità elevata sono **s1/s2** e **s3/s4**.
- **Modelli testati**: Linear Regression (baseline), Ridge, Lasso, ElasticNet — ciascuno su dataset completo e ridotto, con alpha ottimizzato tramite CV.
- **Osservazioni chiave**: Ridge performa meglio sul dataset completo (distribuzione del peso tra correlate); ElasticNet converge a `l1_ratio=1.0` su entrambi i dataset (puro Lasso).
- **Scelta finale data-driven**: Lasso sul dataset selezionato dalla tabella comparativa; alpha ricalcolato sul training set finale.
- **Learning curve**: gap contenuto tra training e validation MSE — modello ben calibrato.
- **Cross-validation** (k=5): R² medio leggermente superiore all'hold-out — la singola suddivisione è leggermente pessimistica con 442 campioni.
- **Esportazione**: modello + scaler salvati con pickle, pronti per la piattaforma MedPredict.

Il modello spiega circa il 43–48% della varianza della progressione del diabete con variabili cliniche di base. Dati longitudinali e biomarcatori aggiuntivi potrebbero incrementare la capacità predittiva in produzione.